In [9]:
import os
import pandas as pd

# ----------------------------------------
# Librerías importadas
# - os: funciones para interactuar con el sistema de archivos
# - pandas: leer CSV en DataFrame y manejar los datos como tabla
# ----------------------------------------

# Rutas de entrada y salida (ajústalas a tu estructura)
CSV_PATH    = "..\\..\\..\\..\\Registros_Clasificados.csv"
OUTPUT_CSV  = "..\\..\\..\\..\\RegistrosOscar_final.csv"

In [10]:
# 1) Cargar el CSV

df = pd.read_csv(CSV_PATH, delimiter=",")

In [11]:
# 2) Función para detectar si existe el label .txt
def check_label(row):
    """
    Determina si existe la carpeta de labels y el .txt asociado para una fila.
    Devuelve 1 si la carpeta labels_x existe y contiene el .txt de la imagen, 
    o 0 en cualquier otro caso.
    """
    base_path  = row['Path']
    img_folder = row['Image_Path']
    img_name   = row['Image_Name']
    
    if pd.isna(img_folder) or pd.isna(img_name):
        return 0
    
    # carpeta labels equivalente
    label_folder = img_folder.replace("images", "labels")
    label_folder_path = os.path.normpath(os.path.join(base_path, label_folder))
    
    # nombre de archivo .txt
    label_file = os.path.splitext(img_name)[0] + ".txt"
    label_file_path = os.path.normpath(os.path.join(label_folder_path, label_file))
    
    return int(os.path.isdir(label_folder_path) and os.path.isfile(label_file_path))

In [18]:
# 3) Función para leer y parsear un archivo YOLO en columnas
def parse_yolo(row):
    """
    Si row['Has_Label']==1, abre el .txt correspondiente y extrae las líneas
    YOLO (<clase> <x_center> <y_center> <width> <height>) en listas.
    Devuelve un Series con cinco columnas de listas; si no existe, devuelve listas vacías.
    """
    if row['Has_Label'] != 1:
        return pd.Series({
            'YOLO_Classes':  [],
            'YOLO_Xcenter':  [],
            'YOLO_Ycenter':  [],
            'YOLO_Width':    [],
            'YOLO_Height':   []
        })
    
    # Ruta al .txt (se asume que ya existe)
    img_folder       = row['Image_Path']
    label_folder     = img_folder.replace("images", "labels")
    label_folder_path= os.path.normpath(os.path.join(row['Path'], label_folder))
    label_file_name  = os.path.splitext(row['Image_Name'])[0] + ".txt"
    label_file_path  = os.path.normpath(os.path.join(label_folder_path, label_file_name))
    
    clases, xcs, ycs, ws, hs = [], [], [], [], []
    with open(label_file_path, 'r') as f:
        for linea in f:
            parts = linea.strip().split()
            print(parts)
            if len(parts) != 5:
                continue
            cls, xc, yc, w, h = parts
            clases.append(int(cls))
            xcs.append(float(xc))
            ycs.append(float(yc))
            ws.append(float(w))
            hs.append(float(h))
    
    return pd.Series({
        'YOLO_Classes':  clases,
        'YOLO_Xcenter':  xcs,
        'YOLO_Ycenter':  ycs,
        'YOLO_Width':    ws,
        'YOLO_Height':   hs
    })


In [19]:
df.iloc[5]

Mail_Date                                    2025-05-06 00:00:00
System_Time                                  2023-09-15 17:12:00
Track_ID                                                      16
Sub_ID                                                         0
Path           ..\Correos\2025\May\06\TRACK ALERT ID 16 at , ...
Image_Name                                                   NaN
Image_Path                                                   NaN
Duration                                                      0s
Latitude                                                  4.6396
Longitude                                               -74.0818
Heading                                                   321.0°
Speed                                                  12.4 km/h
Main_Label                                                    -1
Name: 5, dtype: object

In [23]:
parse_yolo(df.iloc[226])

YOLO_Classes    []
YOLO_Xcenter    []
YOLO_Ycenter    []
YOLO_Width      []
YOLO_Height     []
dtype: object

In [26]:
clases, xcs, ycs, ws, hs = [], [], [], [], []

In [21]:
# 1) Primero calculamos Has_Label para todas las filas
df['Has_Label'] = df.apply(check_label, axis=1)

In [17]:
# 2) Inicializamos las columnas YOLO con listas vacías
for col in ['YOLO_Classes','YOLO_Xcenter','YOLO_Ycenter','YOLO_Width','YOLO_Height']:
    df[col] = [[] for _ in range(len(df))]

# 3) Filtramos sólo las filas que tienen etiqueta y aplicamos parse_yolo
mask = df['Has_Label'] == 1
parsed = df.loc[mask].apply(parse_yolo, axis=1)

# parsed es un DataFrame con las cinco columnas; lo volcamos de vuelta:
df.loc[mask, ['YOLO_Classes','YOLO_Xcenter','YOLO_Ycenter','YOLO_Width','YOLO_Height']] = parsed.values

# 4) Guardamos el resultado
df.to_csv(OUTPUT_CSV, index=False)